

# _Transformers_: Pipelines de tareas habituales de NLP

Francisco Jurado

<francisco.jurado@uam.es>

---

Ejemplos extraídos de https://huggingface.co/learn/nlp-course/chapter1/3

In [ ]:
!pip install --quiet transformers

In [ ]:
from transformers import pipeline
import torch

In [ ]:
# Antes de empezar, definiremos si se ejecutará en CPU o GPU.
# En caso de tener una GPU compatible, se recomienda usarla para acelerar el proceso.
# Para metal, se puede usar "mps" si está disponible, en caso contrario, se usará "cuda", y en último lugar "cpu".
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Qué dispositivo tengo?
print(device)

## Análisis de sentimiento

In [ ]:
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english" ,device=device)
classifier(["I've been waiting for a HuggingFace course my whole life.",
            "I hate this so much!"])

## Classificacion Zero-shot

In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

In [ ]:
classifier(
    "I have a problem with my iphone that needs to be resolved asap!!",
    candidate_labels=["urgent", "not urgent", "phone", "tablet", "computer"],
)

In [ ]:
classifier(
    "Everything was very sudden. The advertisement that aired on television was very unfortunate.",
    candidate_labels=["anger", "joy", "fear", "sadness", "disgust", "surprise"],
)

In [ ]:
classifier(
    "Todo pasó de repente. El anuncio emitido por televisión fue muy desafortunado.",
    candidate_labels=["anger", "joy", "fear", "sadness", "disgust", "surprise"],
)

## Generación de texto

Dado un texto incompleto, devuelve múltiples salidas con las que el texto puede ser completado.

In [ ]:
generator = pipeline("text-generation", model="openai-community/gpt2", device=device)
result = generator("In this course, we will teach you how to")

print(result[0]['generated_text'])

In [ ]:
result = generator(
    "In this course, we will teach you how to",
    max_length=50,
    num_return_sequences=2,
    truncation=True
)

for n, r in enumerate(result):
  print(f'Resultado {n}:')
  print(r['generated_text'])

## Rellenar huecos en blanco (_Mask-filling_)

In [ ]:
unmasker = pipeline("fill-mask", model="distilbert/distilroberta-base" ,device=device)
unmasker("This course will teach you all about <mask> models.", top_k=2)

In [ ]:
unmasker("Madrid is the <mask> of Spain. France is a <mask> of Europe", top_k=2)

In [ ]:
# Named Entity Recognition

ner = pipeline(
    "ner",
    aggregation_strategy="simple",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    device=device
)
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

Como se está usando un modelo predefinido, entrenado para otra tarea y no *fine-tuned*, el warning advierte de que los pesos se han inicializado aleatoriamente.

https://discuss.huggingface.co/t/is-some-weights-of-the-model-were-not-used-warning-normal-when-pre-trained-bert-only-by-mlm/5672

## Respuestas a preguntas (_Question answering_)

In [ ]:
question_answerer = pipeline("question-answering", model="distilbert/distilbert-base-cased-distilled-squad", device=device)
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

## También podemos usar un modelo al que pedirle las diferentes tareas (esta vez sin _Pipeline_)



https://huggingface.co/google/flan-t5-large#TL;DR

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

input_text = "translate English to German: How old are you?"
input_ids = tokenizer(input_text, return_tensors="pt", device=device).input_ids

outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))

In [ ]:
input_text = "Please answer next question: What is the boiling point of Nitrogen?"
input_ids = tokenizer(input_text, return_tensors="pt", device=device).input_ids

outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))

In [ ]:
input_text = "Answer the following question by reasoning step-by-step: \
      The cafeteria has 23 apples. If they used 20 for launch and bought 6 more,\
      how many apples do they have?"
input_ids = tokenizer(input_text, return_tensors="pt", device=device).input_ids

outputs = model.generate(input_ids, max_new_tokens=200)
print(tokenizer.decode(outputs[0]))

In [ ]:
input_text = "Can Geoffrey Hinton have a conversation with George Washintong?\
    Give the rationale before answering."
input_ids = tokenizer(input_text, return_tensors="pt", device=device).input_ids

outputs = model.generate(input_ids, max_new_tokens=200)

# Alternativa:
# encode = tokenizer(input_text, return_tensors="pt", device=device)
# outputs = model.generate(**encode)

print(tokenizer.decode(outputs[0]))